# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [70]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [71]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [102]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
only answer for ticket/flight/related city travel. not any other context.
before booking always ask for confirmation, with details of ticket you are going to book, in deep.
"""

BOOKING TOOL CALLED: Booking flight for Dev2 to London
DETAILS TOOL CALLED: Getting details for ID: None or Name: Dev2
DETAILS TOOL CALLED: Getting details for ID: 1 or Name: None
DETAILS TOOL CALLED: Getting details for ID: 2 or Name: None
DETAILS TOOL CALLED: Getting details for ID: 4 or Name: None


In [101]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [74]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [75]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [76]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [77]:
booking_function = {
    "name": "book_flight",
    "description": "Book a flight for a passenger to a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "passenger_name": {
                "type": "string",
                "description": "The full name of the passenger",
            },
            "destination_city": {
                "type": "string",
                "description": "The city the passenger wants to book a flight to",
            },
        },
        "required": ["passenger_name", "destination_city"],
        "additionalProperties": False
    }
}

In [78]:
details_function = {
    "name": "get_ticket_details",
    "description": "Get flight booking details using a ticket ID or passenger name.",
    "parameters": {
        "type": "object",
        "properties": {
            "ticket_id": {
                "type": "integer",
                "description": "The unique ID of the ticket",
            },
            "passenger_name": {
                "type": "string",
                "description": "The name of the passenger to search for",
            },
        },
        "required": [],
        "additionalProperties": False
    }
}

In [79]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}, {"type": "function", "function": booking_function}, {"type": "function", "function": details_function}]

In [80]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'book_flight',
   'description': 'Book a flight for a passenger to a destination city.',
   'parameters': {'type': 'object',
    'properties': {'passenger_name': {'type': 'string',
      'description': 'The full name of the passenger'},
     'destination_city': {'type': 'string',
      'description': 'The city the passenger wants to book a flight to'}},
    'required': ['passenger_name', 'destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_ticket_details',
   'description': 'Get flight booking details using 

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [81]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [82]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [83]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [84]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [85]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "book_flight":
            arguments = json.loads(tool_call.function.arguments)
            name = arguments.get('passenger_name')
            city = arguments.get('destination_city')
            booking_details = book_flight(name, city)
            responses.append({
                "role": "tool",
                "content": booking_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_ticket_details":
            arguments = json.loads(tool_call.function.arguments)
            ticket_id = arguments.get('ticket_id')
            name = arguments.get('passenger_name')
            details = get_ticket_details(ticket_id=ticket_id, passenger_name=name)
            responses.append({
                "role": "tool",
                "content": details,
                "tool_call_id": tool_call.id
            })
    return responses

In [86]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [105]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [88]:
import sqlite3


In [89]:
import sqlite3
import os

os.makedirs('db', exist_ok=True)
DB = "db/prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    cursor.execute('CREATE TABLE IF NOT EXISTS bookings (id INTEGER PRIMARY KEY AUTOINCREMENT, passenger_name TEXT, city TEXT)')
    conn.commit()

In [90]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [91]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [92]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

def book_flight(passenger_name, destination_city):
    print(f"BOOKING TOOL CALLED: Booking flight for {passenger_name} to {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO bookings (passenger_name, city) VALUES (?, ?)', (passenger_name, destination_city.lower()))
        ticket_id = cursor.lastrowid
        conn.commit()
    return f"Flight successfully booked for {passenger_name} to {destination_city}. Ticket ID: {ticket_id}"

def get_ticket_details(ticket_id=None, passenger_name=None):
    print(f"DETAILS TOOL CALLED: Getting details for ID: {ticket_id} or Name: {passenger_name}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        if ticket_id:
            cursor.execute('SELECT id, passenger_name, city FROM bookings WHERE id = ?', (ticket_id,))
        elif passenger_name:
            cursor.execute('SELECT id, passenger_name, city FROM bookings WHERE passenger_name LIKE ?', (f'%{passenger_name}%',))
        else:
            return "Please provide a ticket ID or a passenger name."
        
        results = cursor.fetchall()
        if not results:
            return "No booking found."
        
        details = []
        for row in results:
            details.append(f"Ticket ID: {row[0]}, Passenger: {row[1]}, Destination: {row[2]}")
        return "\n".join(details)


In [93]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420.5, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [94]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.5'

In [95]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>

## Final Verification Example

Run this to verify the booking tool works.

In [106]:
# Simulate a chat that results in a booking and then a query
print("--- BOOKING ---")
booking_resp = chat("I'd like to book a flight to Tokyo for Alice", [])
print(booking_resp)

import re
match = re.search(r'Ticket ID: (\d+)', booking_resp)
if match:
    ticket_id = int(match.group(1))
    print(f"\n--- QUERYING BY ID {ticket_id} ---")
    print(chat(f"What are the details for ticket {ticket_id}?", [{"role": "assistant", "content": booking_resp}]))

print("\n--- QUERYING BY NAME ---")
print(chat("Can you show me the booking for Alice?", []))


--- BOOKING ---
Could you please confirm that you want to book a flight to Tokyo for Alice, and provide the full name exactly as it should appear on the ticket?

--- QUERYING BY NAME ---
DETAILS TOOL CALLED: Getting details for ID: None or Name: Alice
Alice has two bookings, both to Tokyo. Would you like details on a specific booking?
